# 7. Azure AI + RAG Architecture

## 1. What is RAG?

**Retrieval-Augmented Generation (RAG)** combines:

1. **Retrieval** — retrieve relevant information from enterprise data.
2. **Generation** — provide that information to an LLM to generate the answer.

Instead of asking the LLM to answer only from its pretrained knowledge:

```text
User Question
     ↓
     LLM
     ↓
Answer
```

we use:

```text
User Question
     ↓
Retrieve relevant enterprise data
     ↓
Context + Question
     ↓
LLM
     ↓
Grounded Answer
```

---

# 2. Azure RAG Architecture

A typical Azure enterprise RAG architecture:

```text
                         USER
                           │
                           ▼
                    Web / Teams / App
                           │
                           ▼
                     API / Backend
                           │
                           ▼
                    Query Processing
                           │
                           ▼
                  Azure AI Search
                           │
              ┌────────────┼────────────┐
              ▼            ▼            ▼
          Keyword       Vector       Semantic
           Search       Search        Ranking
              │            │            │
              └────────────┼────────────┘
                           ▼
                    Relevant Chunks
                           │
                           ▼
                  Prompt + Context
                           │
                           ▼
                    Azure OpenAI
                           │
                           ▼
                     Final Answer
```

---

# 3. Complete RAG Lifecycle

There are two major phases:

```text
        ┌─────────────────────────┐
        │     INDEXING PHASE      │
        └─────────────────────────┘

Documents
   ↓
Document Intelligence / Parser
   ↓
Cleaning & Normalization
   ↓
Chunking
   ↓
Embeddings
   ↓
Azure AI Search
   ↓
Search Index


        ┌─────────────────────────┐
        │      QUERY PHASE       │
        └─────────────────────────┘

User Question
   ↓
Query Processing
   ↓
Azure AI Search
   ↓
Retrieve Top-K
   ↓
Context Construction
   ↓
Azure OpenAI
   ↓
Grounded Response
```

---

# 4. Indexing Pipeline

Suppose the enterprise has:

```text
SharePoint
PDFs
Word documents
Invoices
HR policies
Product documents
```

The ingestion pipeline can be:

```text
Enterprise Documents
        │
        ▼
Blob Storage / SharePoint
        │
        ▼
Document Intelligence
        │
        ▼
Text + Tables + Structure
        │
        ▼
Chunking
        │
        ▼
Embedding Model
        │
        ▼
Azure AI Search Index
```

---

# 5. Document Processing

For complex PDFs, use **Azure Document Intelligence**.

It can extract:

- Text
- Tables
- Key-value pairs
- Layout
- OCR content
- Document structure

Example:

```text
PDF
 │
 ├── Page 1 → Text
 ├── Page 2 → Table
 ├── Page 3 → Scanned Image → OCR
 └── Page 4 → Text
```

Preserve metadata such as:

```json
{
  "document_id": "policy_001",
  "page": 4,
  "section": "Leave Policy",
  "document_type": "HR_POLICY",
  "content": "Employees are entitled..."
}
```

Metadata becomes valuable during retrieval and citation.

---

# 6. Chunking

You generally shouldn't embed an entire 200-page document as one vector.

Instead:

```text
200-page PDF
      ↓
Document structure
      ↓
Sections / paragraphs / tables
      ↓
Chunks
```

For example:

```text
Document
 ├── Chunk 1 → Introduction
 ├── Chunk 2 → Eligibility
 ├── Chunk 3 → Leave Policy
 ├── Chunk 4 → Carry Forward
 └── Chunk 5 → Exceptions
```

For enterprise documents, **structure-aware / semantic chunking** is often preferable to blindly splitting every N characters.

---

# 7. Embeddings

Each chunk is converted into a vector.

```text
Chunk
 ↓
Embedding Model
 ↓
[0.12, 0.84, -0.31, ...]
```

The vector is stored in Azure AI Search.

At query time:

```text
User Question
 ↓
Same embedding model
 ↓
Query Vector
 ↓
Vector Search
```

The embedding model used for documents and queries needs compatible vector dimensions.

---

# 8. Azure AI Search Index

A simplified index could contain:

| Field | Purpose |
|---|---|
| `id` | Unique chunk ID |
| `content` | Chunk text |
| `contentVector` | Embedding vector |
| `documentId` | Source document |
| `pageNumber` | Source page |
| `section` | Section name |
| `documentType` | Metadata |
| `department` | Metadata |
| `securityGroup` | Access control metadata |

Example:

```json
{
  "id": "policy_001_chunk_04",
  "content": "Employees can carry forward...",
  "contentVector": [0.12, 0.42, -0.18],
  "documentId": "policy_001",
  "pageNumber": 12,
  "section": "Carry Forward",
  "department": "HR"
}
```

---

# 9. Query Phase

User asks:

> "Can I carry forward unused vacation days?"

The application sends the query to the retrieval layer.

```text
User Question
      ↓
Query Processing
      ↓
Azure AI Search
```

---

# 10. Hybrid Search ⭐⭐⭐⭐⭐

For enterprise RAG, a strong approach is:

```text
             User Query
                  │
        ┌─────────┴─────────┐
        ▼                   ▼
   Keyword Search      Vector Search
        │                   │
        └─────────┬─────────┘
                  ▼
            Hybrid Results
                  │
                  ▼
          Semantic Ranking
                  │
                  ▼
               Top-K
```

### Keyword search

Good for:

- Employee IDs
- Policy numbers
- Product codes
- Exact terminology

### Vector search

Good for:

- Semantic similarity
- Natural language questions
- Different wording with the same meaning

### Hybrid search

Combines both.

---

# 11. Metadata Filtering

Suppose the enterprise has policies for multiple countries.

User:

> "What is the leave policy in India?"

You can retrieve with filters such as:

```text
country = 'India'
AND
documentType = 'HR_POLICY'
```

Then perform semantic/vector/keyword retrieval within that scope.

This is important for **precision and data isolation**.

---

# 12. Access Control in RAG ⭐⭐⭐⭐⭐

This is a major production consideration.

Suppose:

```text
User A → HR documents
User B → Finance documents
```

You shouldn't simply retrieve the most relevant documents globally.

Store security metadata:

```text
document
 ├── department = HR
 ├── allowed_groups = HR_TEAM
```

Then:

```text
User
 ↓
Identity
 ↓
Authorization
 ↓
Search Filter
 ↓
Allowed Documents
 ↓
LLM
```

The LLM should **not** be responsible for enforcing authorization.

---

# 13. Context Construction

Suppose retrieval returns:

```text
Chunk 1
Chunk 2
Chunk 3
```

The application constructs:

```text
System Instruction
+
User Question
+
Retrieved Context
```

Example:

```text
SYSTEM:
Answer only using the supplied context.
If the answer isn't present, say you don't have enough information.

CONTEXT:
[Chunk 1]
[Chunk 2]
[Chunk 3]

QUESTION:
Can I carry forward unused vacation days?
```

Then:

```text
Prompt
 ↓
Azure OpenAI
```

---

# 14. Grounded Generation

The LLM should generate its response based on retrieved enterprise information.

```text
Retrieved Context
       +
User Question
       ↓
Azure OpenAI
       ↓
Grounded Response
```

Example:

```text
Context:
"Employees can carry forward up to 5 unused vacation days."

Question:
"How many vacation days can I carry forward?"

Answer:
"Employees can carry forward up to 5 unused vacation days."
```

---

# 15. Citations

For enterprise RAG, maintain source metadata.

Instead of:

> "You can carry forward 5 days."

return:

> "You can carry forward up to 5 unused vacation days. Source: Leave Policy, page 12."

Architecture:

```text
Search Result
 ├── content
 ├── documentId
 ├── pageNumber
 └── section
          ↓
       LLM Answer
          +
       Citation
```

This improves:

- Trust
- Traceability
- Auditability
- User confidence

---

# 16. Azure Components in RAG

| Azure Component | Role in RAG |
|---|---|
| **Azure Blob Storage** | Store documents |
| **SharePoint** | Enterprise document source |
| **Azure Document Intelligence** | Extract text/structure/OCR |
| **Embedding Model** | Convert text to vectors |
| **Azure AI Search** | Index + retrieve |
| **Azure OpenAI** | Generate response |
| **Microsoft Entra ID** | Authentication/identity |
| **Managed Identity** | Service-to-service authentication |
| **Key Vault** | Secret management |
| **Azure Functions** | Serverless processing |
| **API Management** | API gateway/security |
| **Azure Monitor** | Monitoring |
| **Application Insights** | Application telemetry |

---

# 17. RAG Architecture with Security

A production architecture can look like:

```text
                         User
                           │
                           ▼
                    Web / Teams
                           │
                           ▼
                  API Management
                           │
                           ▼
                     AI Backend
                           │
                    Entra ID Auth
                           │
                           ▼
                   RAG Orchestrator
                           │
                           ▼
                  Azure AI Search
                           │
                 Security Filtering
                           │
                           ▼
                  Relevant Chunks
                           │
                           ▼
                    Azure OpenAI
                           │
                           ▼
                  Content Safety
                           │
                           ▼
                       Answer
```

Supporting services:

```text
Blob Storage ──→ Document Intelligence
                      │
                      ▼
                   Chunking
                      │
                      ▼
                 Embeddings
                      │
                      ▼
               Azure AI Search
```

---

# 18. RAG with LangChain

You can still use your familiar LangChain architecture.

```text
User
 ↓
LangChain
 ↓
Azure AI Search Retriever
 ↓
Retrieved Documents
 ↓
Prompt
 ↓
Azure OpenAI
 ↓
Answer
```

For example:

```python
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint="YOUR_ENDPOINT",
    azure_deployment="YOUR_DEPLOYMENT",
    api_version="YOUR_API_VERSION"
)
```

Then the retriever:

```text
Azure AI Search
       ↓
Retriever
       ↓
Documents
       ↓
LLM
```

The exact LangChain integration/API should be aligned with the current package versions used in the project.

---

# 19. RAG with LangGraph

For a more complex enterprise system:

```text
                    START
                      │
                      ▼
                 Query Router
                      │
              ┌───────┴────────┐
              ▼                ▼
          Search RAG       Direct LLM
              │
              ▼
         Azure AI Search
              │
              ▼
       Retrieve Documents
              │
              ▼
         Grade Results
              │
         ┌────┴────┐
         │         │
      Relevant   Not Relevant
         │         │
         ▼         ▼
      Generate   Rewrite Query
         │         │
         │         └──────→ Search
         ▼
      Validate
         │
         ▼
        END
```

This is where **Agentic RAG** becomes more sophisticated than a simple RAG chain.

---

# 20. RAG Evaluation

Don't evaluate only the final answer.

Evaluate multiple layers.

### Retrieval

```text
Recall@K
Precision@K
MRR
NDCG
```

### Generation

```text
Faithfulness
Groundedness
Answer Relevance
```

### System

```text
Latency
Token Usage
Cost
Failure Rate
```

Example:

```text
Question
   ↓
Retriever
   ↓
Recall@K
   ↓
Context
   ↓
LLM
   ↓
Groundedness
   ↓
Final Answer
```

---

# 21. Common RAG Problems

### Problem 1 — Correct document not retrieved

Investigate:

```text
Chunking
Embedding
Hybrid Search
Top-K
Filters
Semantic Ranking
```

### Problem 2 — Correct document retrieved but wrong answer

Investigate:

```text
Prompt
Context construction
Context ordering
LLM
Grounding instructions
```

### Problem 3 — Hallucination

Use:

```text
Better retrieval
+
Grounded prompts
+
Output validation
+
Groundedness evaluation
+
Content Safety
```

### Problem 4 — Too much context

Use:

```text
Top-K optimization
Reranking
Chunk optimization
Context compression
```

---

# 22. Large PDF Example

Suppose:

> **200-page PDF, important information on page 2 and related information on page 30.**

A robust Azure pipeline:

```text
200-page PDF
      ↓
Document Intelligence
      ↓
Structure + Metadata
      ↓
Structure-aware Chunking
      ↓
Embeddings
      ↓
Azure AI Search
      ↓
Hybrid Retrieval
      ↓
Retrieve relevant chunks
      ↓
Semantic Ranking
      ↓
Context Assembly
      ↓
Azure OpenAI
      ↓
Answer
```

Because chunks retain metadata such as:

```text
document_id
page_number
section
```

the system can retrieve information from **both page 2 and page 30** when both are relevant.

---

# 23. Production-Grade Considerations

A production Azure RAG system should consider:

| Area | Considerations |
|---|---|
| **Security** | Entra ID, RBAC, document-level access |
| **Retrieval** | Hybrid search, filters, reranking |
| **Data** | Chunking, metadata, document versions |
| **LLM** | Model selection, token limits |
| **Reliability** | Retry, timeout, fallback |
| **Performance** | Caching, parallel retrieval |
| **Cost** | Tokens, embedding calls, search |
| **Observability** | Logs, traces, metrics |
| **Evaluation** | Retrieval + generation metrics |
| **Safety** | Content Safety, prompt injection |
| **Scalability** | Search capacity, model quotas |
| **Auditability** | Citations, source tracking |

---

# 24. Interview Questions

### Q1. Design an enterprise RAG solution using Azure.

> "I would use Blob Storage or SharePoint as document sources, Document Intelligence for complex document extraction, an embedding model for vectorization, Azure AI Search for vector/hybrid retrieval and semantic ranking, and Azure OpenAI for grounded generation. I would use Entra ID and Managed Identity for security, Key Vault for secrets where required, and Azure Monitor/Application Insights for observability."

### Q2. Why Azure AI Search instead of just a vector database?

> "Azure AI Search provides more than vector search. It supports keyword search, vector search, hybrid search, filtering and semantic ranking, making it suitable for enterprise search and RAG workloads."

### Q3. How do you prevent unauthorized documents from entering the prompt?

> "I enforce authorization before retrieval and apply security filters to the search query based on the user's identity and permissions. The LLM itself is not responsible for access control."

### Q4. How do you improve poor RAG accuracy?

> "I first separate retrieval problems from generation problems. I evaluate chunking, embeddings, metadata filters, hybrid retrieval, top-K and ranking, then evaluate context construction and generation separately."

### Q5. How do you handle a large PDF?

> "I would process it asynchronously, extract structured content with Document Intelligence, use structure-aware chunking, preserve page and section metadata, index the chunks in Azure AI Search, and retrieve only relevant context at query time."

---

# 25. One-Line Architecture

For an interview whiteboard:

```text
Documents
   ↓
Blob / SharePoint
   ↓
Document Intelligence
   ↓
Chunk + Metadata
   ↓
Embeddings
   ↓
Azure AI Search
   ↓
Hybrid / Vector + Semantic Ranking
   ↓
Top-K Context
   ↓
Azure OpenAI
   ↓
Content Safety / Validation
   ↓
Grounded Answer
```

This is the core **Azure enterprise RAG architecture**.
